# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading, exploring, and preprocessing the [FAIR^2 Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"Dataset Title: {metadata.name}\n\nDescription: {metadata.description}\n")

## 2. Data Overview
Review available record sets, fields, and their `@id` values so we know what data can be loaded.

We'll enumerate all record sets and their fields, listing their `@id`s. _All entities_ (record sets, fields, columns) are referenced by their `@id`, as required.

In [ ]:
# List all record sets (`cr:RecordSet`) and their fields
record_sets = []
print('Record sets available in dataset:')
for rs in metadata.record_set:
    print(f"\nRecordSet: {rs.name}")
    print(f"  @id         : {rs.id}")
    print(f"  Description : {getattr(rs, 'description', '')}")
    # Collect record set ids
    record_sets.append(rs.id)
    if hasattr(rs, 'field') and rs.field:
        print(f"  Fields:")
        for field in rs.field:
            print(f"    - Name: {field.name}")
            print(f"      @id : {field.id}")
            print(f"      Type: {field.data_type}")
            print(f"      Description: {getattr(field, 'description', '')}")


## 3. Data Extraction
Load data from the primary record sets into Pandas DataFrames for analysis.

> _Use only the `@id` for each record set. We extract all record sets for demonstration. For large files, consider extracting only those you need._

In [ ]:
# Extract data from each record set into a DataFrame (using each record set's @id)
dataframes = {}
for record_set_id in record_sets:
    print(f"Loading: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:  # Only load if there are records
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"  Columns: {df.columns.tolist()}")
        print(df.head(2))
    else:
        print("  No records available.")

# Use the first available record set for further analysis
if dataframes:
    main_record_set_id = list(dataframes.keys())[0]
    print(f"\nMain analysis record set: {main_record_set_id}")
    print(dataframes[main_record_set_id].head())
else:
    print('No dataframes were loaded. Please check the dataset.')

## 4. Exploratory Data Analysis (EDA)
Let's analyze the main record set. We'll select a numeric field using its `@id`, filter and analyze the data, and demonstrate normalization/grouping using only the correct variable/data reference.

> **Note:** Please consult the output in Section 2 to choose available numeric fields (e.g., quantities, coverage, counts, abundances, MW, etc.). Edit the `numeric_field_id` variable to try out analyses on other fields.


In [ ]:
# Select a numeric field (update its `@id` as per Section 2 output, e.g. 'coverage', 'MW', etc.)
main_df = dataframes[main_record_set_id]

# Choose a column (field) id for analysis; update as appropriate.
numeric_field_id = None
# Find a candidate numeric field
for col in main_df.columns:
    # Try to pick a likely numeric field for demonstration
    if ('coverage' in col.lower() or 'mw' in col.lower() or 'abundance' in col.lower() or 'count' in col.lower()):
        numeric_field_id = col
        break

if numeric_field_id is None:
    numeric_field_id = main_df.columns[0]  # fallback
print(f"Using numeric field: {numeric_field_id}")

# Ensure it's float
main_df[numeric_field_id] = pd.to_numeric(main_df[numeric_field_id], errors='coerce')

# Filter records with values above a threshold
threshold = main_df[numeric_field_id].dropna().quantile(0.25)  # for demo, filter upper quartile
filtered_df = main_df[main_df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
print(filtered_df.head())

# Normalize the numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Try to group by categorical column (e.g. 'description', 'protein', or similar; pick from columns)
group_field_id = None
for col in main_df.columns:
    if 'desc' in col.lower() or 'group' in col.lower() or 'protei' in col.lower():
        group_field_id = col
        break

if group_field_id:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame('mean').sort_values('mean', ascending=False)
    print(f"\nGrouped data by {group_field_id} (mean of {numeric_field_id}):")
    print(grouped_df.head())
else:
    print('No grouping field found.')

## 5. Visualization
Visualize distributions or relationships. We'll plot the distribution of the chosen numeric field and (if grouped data exists) the top groups by mean value.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot distribution of the numeric field
plt.figure(figsize=(8,4))
sns.histplot(filtered_df[numeric_field_id].dropna(), kde=True)
plt.title(f'Distribution of {numeric_field_id}')
plt.xlabel(numeric_field_id)
plt.ylabel('Frequency')
plt.show()

# If grouped data exists, plot top 10 groups
if 'grouped_df' in locals():
    plt.figure(figsize=(10,4))
    grouped_df_sorted = grouped_df.head(10)
    sns.barplot(x=grouped_df_sorted.index, y='mean', data=grouped_df_sorted, palette='viridis')
    plt.title(f'Top {group_field_id} groups by mean {numeric_field_id}')
    plt.ylabel(f'Mean {numeric_field_id}')
    plt.xlabel(f'{group_field_id}')
    plt.xticks(rotation=30, ha='right')
    plt.show()

## 6. Conclusion
In this notebook, we loaded the FAIR^2 Mass Spectrometry dataset via its Croissant schema, examined record set structures and their field `@id`s, extracted data dynamically via `mlcroissant`, and performed basic filtering, normalization, and visual exploration using only structure-aware field identifiers. This approach can be generalized for any Croissant dataset by adjusting references to relevant `@id` identifiers.

Further domain-specific analysis, such as biomarker discovery, differential protein expression, or mass-spectra feature engineering, can be built on top of this foundation.
